# §1.6 アファイン接続と共変微分 - 曲がった空間での微分

## 1. 概要

- **この節で学ぶこと**: アファイン接続、共変微分、平行移動、曲率
- **前提知識**: ベクトル場、リーマン計量、Christoffel記号
- **情報幾何との関連**: **α-接続、e-接続とm-接続の双対性**

## 2. 直感的理解

### なぜ「普通の微分」ではダメなのか

- ユークリッド空間: ベクトルを平行移動しても成分は変わらない
- 曲がった空間: 基底自体が場所によって変わる
- → ベクトルの「本当の変化」と「座標系の変化」を区別する必要

### 共変微分のイメージ

- 「座標系の変化を差し引いた、真のベクトル場の変化率」
- 平行移動: 共変微分がゼロとなる移動

### 地球表面での例え

- 北極から赤道へベクトル（東向き）を平行移動
- 経路によって最終的なベクトルが異なる！
- これが「曲率」の表れ

### 情報幾何での重要性

- **α-接続**: パラメータ α で特徴づけられる接続の族
- **e-接続** (α=1): 指数型分布族で自然な接続
- **m-接続** (α=-1): 混合族で自然な接続
- **双対性**: e-接続と m-接続はFisher計量に関して双対

## 3. 数学的定義

### 3.1 アファイン接続の定義

多様体 $M$ 上の**アファイン接続** $\nabla$ とは、ベクトル場 $X, Y$ に対して
$$\nabla: \mathfrak{X}(M) \times \mathfrak{X}(M) \to \mathfrak{X}(M)$$
$$\nabla: (X, Y) \mapsto \nabla_X Y$$
で、以下を満たすもの:

1. $\nabla_{fX+gY} Z = f\nabla_X Z + g\nabla_Y Z$
2. $\nabla_X (Y + Z) = \nabla_X Y + \nabla_X Z$
3. $\nabla_X (fY) = (Xf)Y + f\nabla_X Y$ (ライプニッツ則)

### 3.2 Christoffel記号

座標基底 $\{\partial_i\}$ に対する接続係数:
$$\nabla_{\partial_i} \partial_j = \Gamma^k_{ij} \partial_k$$

ベクトル場 $Y = Y^j \partial_j$ の共変微分:
$$\nabla_X Y = X^i \left( \frac{\partial Y^k}{\partial x^i} + \Gamma^k_{ij} Y^j \right) \partial_k$$

### 3.3 Levi-Civita接続

リーマン多様体で唯一の、以下を満たす接続:
- **計量整合性**: $Xg(Y, Z) = g(\nabla_X Y, Z) + g(Y, \nabla_X Z)$
- **捩れなし**: $\nabla_X Y - \nabla_Y X = [X, Y]$

Christoffel記号の公式:
$$\Gamma^k_{ij} = \frac{1}{2} g^{kl} \left( \frac{\partial g_{il}}{\partial x^j} + \frac{\partial g_{jl}}{\partial x^i} - \frac{\partial g_{ij}}{\partial x^l} \right)$$

### 3.4 α-接続（情報幾何）

Fisher計量 $g$ に対して、**α-接続** $\nabla^{(\alpha)}$ を定義:
$$\Gamma^{(\alpha)k}_{ij} = \Gamma^{(0)k}_{ij} - \frac{\alpha}{2} T_{ij}^k$$

ここで $\Gamma^{(0)}$ はLevi-Civita接続、$T$ は歪度テンソル。

重要な特殊ケース:
- $\alpha = 0$: Levi-Civita接続
- $\alpha = 1$: **e-接続**（指数接続）
- $\alpha = -1$: **m-接続**（混合接続）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import odeint

plt.rcParams['figure.figsize'] = (10, 8)

class AffineConnection:
    """アファイン接続のクラス"""
    def __init__(self, christoffel_func, dim=2):
        """
        christoffel_func: 点 p での Γ^k_{ij} を返す関数
                          christoffel_func(p) -> array of shape (dim, dim, dim)
        """
        self.christoffel_func = christoffel_func
        self.dim = dim
    
    def christoffel(self, p):
        """点 p での Christoffel 記号 Γ^k_{ij}"""
        return self.christoffel_func(p)
    
    def covariant_derivative(self, p, X, Y, dY):
        """
        共変微分 (∇_X Y)^k = X^i (∂Y^k/∂x^i + Γ^k_{ij} Y^j)
        
        p: 点
        X: 方向ベクトル
        Y: ベクトル場の値
        dY: ベクトル場の偏微分 (∂Y^k/∂x^i)
        """
        Gamma = self.christoffel(p)
        result = np.zeros(self.dim)
        
        for k in range(self.dim):
            for i in range(self.dim):
                result[k] += X[i] * dY[i, k]
                for j in range(self.dim):
                    result[k] += X[i] * Gamma[k, i, j] * Y[j]
        
        return result
    
    def geodesic_ode(self, state, t):
        """測地線の微分方程式"""
        p = state[:self.dim]
        v = state[self.dim:]
        
        Gamma = self.christoffel(p)
        
        dp = v
        dv = np.zeros(self.dim)
        
        for k in range(self.dim):
            for i in range(self.dim):
                for j in range(self.dim):
                    dv[k] -= Gamma[k, i, j] * v[i] * v[j]
        
        return np.concatenate([dp, dv])
    
    def geodesic(self, p0, v0, t_span, n_points=100):
        """測地線を計算"""
        t = np.linspace(t_span[0], t_span[1], n_points)
        state0 = np.concatenate([p0, v0])
        
        solution = odeint(self.geodesic_ode, state0, t)
        
        return t, solution[:, :self.dim], solution[:, self.dim:]

## 4. 可視化

### 4.1 球面上の平行移動

In [ ]:
def visualize_parallel_transport_sphere():
    """球面上の平行移動を可視化"""
    fig = plt.figure(figsize=(12, 5))
    
    # 球面を描画
    u = np.linspace(0, 2*np.pi, 50)
    v = np.linspace(0, np.pi, 30)
    U, V = np.meshgrid(u, v)
    X = np.sin(V) * np.cos(U)
    Y = np.sin(V) * np.sin(U)
    Z = np.cos(V)
    
    # 左図：3D球面
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(X, Y, Z, alpha=0.3, color='cyan')
    
    # 経路1: 北極 → 経度0で赤道 → 経度90°で北極
    # 経路2: 北極 → 経度90°で赤道 → 経度0で北極（逆経路）
    
    # 経路をプロット（三角形）
    # 北極
    p_north = np.array([0, 0, 1])
    # 赤道（経度0）
    p_eq_0 = np.array([1, 0, 0])
    # 赤道（経度90°）
    p_eq_90 = np.array([0, 1, 0])
    
    # 測地線（大円）
    t = np.linspace(0, np.pi/2, 50)
    
    # 北極 → 赤道(0)
    path1_x = np.sin(t)
    path1_y = np.zeros_like(t)
    path1_z = np.cos(t)
    ax1.plot(path1_x, path1_y, path1_z, 'r-', linewidth=3, label='経路1')
    
    # 赤道(0) → 赤道(90)
    path2_x = np.cos(t)
    path2_y = np.sin(t)
    path2_z = np.zeros_like(t)
    ax1.plot(path2_x, path2_y, path2_z, 'g-', linewidth=3, label='経路2')
    
    # 赤道(90) → 北極
    path3_x = np.zeros_like(t)
    path3_y = np.sin(np.pi/2 - t)
    path3_z = np.cos(np.pi/2 - t)
    ax1.plot(path3_x, path3_y, path3_z, 'b-', linewidth=3, label='経路3')
    
    # 頂点をマーク
    ax1.scatter([0, 1, 0], [0, 0, 1], [1, 0, 0], s=100, c='black')
    
    # ベクトルの平行移動を示す矢印
    # 北極での初期ベクトル（経度0方向 = +x方向）
    ax1.quiver(0, 0, 1, 0.3, 0, 0, color='red', arrow_length_ratio=0.3, linewidth=2)
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('球面上の三角形経路\n（測地三角形）')
    ax1.legend()
    
    # 右図：平行移動の結果
    ax2 = fig.add_subplot(122)
    
    ax2.text(0.1, 0.9, '【平行移動の結果】', fontsize=14, transform=ax2.transAxes)
    ax2.text(0.1, 0.8, '初期ベクトル: 北極で東向き（+x方向）', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.1, 0.7, '', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.1, 0.6, '経路1（北極→赤道(0)）:', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.15, 0.55, '→ 赤道で南向き', fontsize=10, transform=ax2.transAxes)
    ax2.text(0.1, 0.45, '経路2（赤道を東へ）:', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.15, 0.4, '→ 赤道(90°)で南向きのまま', fontsize=10, transform=ax2.transAxes)
    ax2.text(0.1, 0.3, '経路3（赤道(90°)→北極）:', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.15, 0.25, '→ 北極で+y方向', fontsize=10, transform=ax2.transAxes)
    ax2.text(0.1, 0.1, '結果: ベクトルが90°回転！', fontsize=12, color='red', 
             fontweight='bold', transform=ax2.transAxes)
    ax2.text(0.1, 0.02, '（これが球面の曲率の表れ）', fontsize=11, transform=ax2.transAxes)
    
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("【ホロノミー】")
    print("閉じた経路に沿って平行移動すると、ベクトルは元に戻らない場合がある。")
    print("回転角 = 経路が囲む面積 × ガウス曲率")
    print(f"この例: 回転角 = (1/8 × 4π) × 1 = π/2")

visualize_parallel_transport_sphere()

### 4.2 正規分布多様体での測地線

In [ ]:
def visualize_gaussian_geodesics():
    """正規分布多様体での測地線（Levi-Civita接続）"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 正規分布のFisher計量とChristoffel記号
    # g = diag(1/σ², 2/σ²)
    # Γ^μ_{μσ} = Γ^μ_{σμ} = 0
    # Γ^σ_{μμ} = 0
    # Γ^μ_{σσ} = 0
    # Γ^σ_{σσ} = -1/σ
    # Γ^μ_{μμ} = 0, Γ^σ_{μμ} = 1/σ³ × 1/σ² × σ² = 1/σ (from metric compatibility)
    
    def christoffel_gaussian(p):
        """正規分布のChristoffel記号（Levi-Civita接続）"""
        mu, sigma = p
        sigma = max(sigma, 0.1)  # 安定性のため
        
        Gamma = np.zeros((2, 2, 2))  # Γ^k_{ij}
        
        # 非ゼロ成分
        Gamma[0, 0, 1] = -1/sigma  # Γ^μ_{μσ}
        Gamma[0, 1, 0] = -1/sigma  # Γ^μ_{σμ}
        Gamma[1, 0, 0] = 1/sigma   # Γ^σ_{μμ}
        Gamma[1, 1, 1] = -1/sigma  # Γ^σ_{σσ}
        
        return Gamma
    
    conn = AffineConnection(christoffel_gaussian)
    
    # 左図：様々な初期条件からの測地線
    ax1 = axes[0]
    
    # Fisher計量楕円を背景に
    theta = np.linspace(0, 2*np.pi, 100)
    for mu_c in np.linspace(-1, 2, 4):
        for sigma_c in [0.5, 1.0, 1.5]:
            scale = 0.1
            ellipse_x = sigma_c * np.cos(theta) * scale + mu_c
            ellipse_y = sigma_c / np.sqrt(2) * np.sin(theta) * scale + sigma_c
            ax1.plot(ellipse_x, ellipse_y, color='gray', linewidth=0.5, alpha=0.3)
    
    # 測地線を計算
    initial_conditions = [
        ([0, 1], [1, 0]),      # μ方向
        ([0, 1], [0, 0.5]),    # σ方向
        ([0, 1], [1, 0.3]),    # 斜め
        ([0, 1], [1, -0.3]),   # 斜め（下向き）
        ([-1, 0.8], [1, 0.2]),
    ]
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(initial_conditions)))
    
    for (p0, v0), color in zip(initial_conditions, colors):
        try:
            t, path, vel = conn.geodesic(np.array(p0), np.array(v0), [0, 2], n_points=100)
            # σ > 0 の部分のみプロット
            valid = path[:, 1] > 0.1
            ax1.plot(path[valid, 0], path[valid, 1], color=color, linewidth=2)
            ax1.plot(p0[0], p0[1], 'o', color=color, markersize=8)
            ax1.arrow(p0[0], p0[1], v0[0]*0.2, v0[1]*0.2, head_width=0.05, 
                      head_length=0.02, fc=color, ec=color)
        except:
            pass
    
    ax1.set_xlabel('μ')
    ax1.set_ylabel('σ')
    ax1.set_title('正規分布多様体での測地線\n（Levi-Civita接続）')
    ax1.set_xlim(-2, 3)
    ax1.set_ylim(0.1, 2)
    ax1.grid(True, alpha=0.3)
    
    # 右図：2点間の測地線
    ax2 = axes[1]
    
    # 2点を指定
    p1 = np.array([0, 1])
    p2 = np.array([2, 0.5])
    
    ax2.plot(p1[0], p1[1], 'go', markersize=15, label='始点 N(0, 1²)')
    ax2.plot(p2[0], p2[1], 'ro', markersize=15, label='終点 N(2, 0.5²)')
    
    # 直線（ユークリッド）
    t_line = np.linspace(0, 1, 50)
    line = p1[:, None] + t_line * (p2 - p1)[:, None]
    ax2.plot(line[0], line[1], 'b--', linewidth=2, label='直線（ユークリッド）')
    
    # 測地線（初速度を調整して終点に到達）
    v0_approx = np.array([1.5, -0.3])  # 概算の初速度
    t, path, vel = conn.geodesic(p1, v0_approx, [0, 1.5], n_points=100)
    valid = path[:, 1] > 0.1
    ax2.plot(path[valid, 0], path[valid, 1], 'r-', linewidth=2, label='測地線（Fisher計量）')
    
    ax2.set_xlabel('μ')
    ax2.set_ylabel('σ')
    ax2.set_title('直線 vs 測地線\n（測地線は小さいσの領域を避ける）')
    ax2.set_xlim(-0.5, 3)
    ax2.set_ylim(0.1, 1.5)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("【観察】")
    print("- 測地線は『Fisher距離を最短にする経路』")
    print("- σが小さい領域はFisher情報が大きい → 同じ変化でも『遠い』")
    print("- そのため測地線はσが小さい領域を避けて迂回する")

visualize_gaussian_geodesics()

### 4.3 α-接続の違い（e-接続 vs m-接続）

In [ ]:
def visualize_alpha_connections():
    """α-接続による測地線の違い"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 指数型分布族（正規分布）でのα-接続
    # α = 1: e-geodesic (自然パラメータで直線)
    # α = 0: Levi-Civita geodesic
    # α = -1: m-geodesic (期待値パラメータで直線)
    
    # ベルヌーイ分布の場合を考える（1次元）
    # p ∈ (0, 1)
    # 自然パラメータ: θ = log(p/(1-p))
    # 期待値パラメータ: η = p
    
    def p_to_theta(p):
        """期待値 → 自然パラメータ"""
        p = np.clip(p, 0.01, 0.99)
        return np.log(p / (1 - p))
    
    def theta_to_p(theta):
        """自然パラメータ → 期待値"""
        return 1 / (1 + np.exp(-theta))
    
    # 2点: p1 = 0.2, p2 = 0.8
    p1, p2 = 0.2, 0.8
    theta1, theta2 = p_to_theta(p1), p_to_theta(p2)
    
    t = np.linspace(0, 1, 100)
    
    # 左図：期待値パラメータ空間
    ax1 = axes[0]
    
    # m-geodesic (直線 in η)
    p_m = p1 + t * (p2 - p1)
    ax1.plot(t, p_m, 'b-', linewidth=2, label='m-測地線 (α=-1)\nη空間で直線')
    
    # e-geodesic (直線 in θ → 変換)
    theta_e = theta1 + t * (theta2 - theta1)
    p_e = theta_to_p(theta_e)
    ax1.plot(t, p_e, 'r-', linewidth=2, label='e-測地線 (α=1)\nθ空間で直線')
    
    # 中点をマーク
    ax1.plot(0.5, (p1+p2)/2, 'b^', markersize=12)  # m-中点
    ax1.plot(0.5, theta_to_p((theta1+theta2)/2), 'rv', markersize=12)  # e-中点
    
    ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel('t')
    ax1.set_ylabel('p（確率）')
    ax1.set_title('期待値パラメータ空間での測地線\nBer(0.2) → Ber(0.8)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # 中図：自然パラメータ空間
    ax2 = axes[1]
    
    # e-geodesic (直線 in θ)
    ax2.plot(t, theta_e, 'r-', linewidth=2, label='e-測地線 (α=1)\nθ空間で直線')
    
    # m-geodesic (変換)
    theta_m = p_to_theta(p_m)
    ax2.plot(t, theta_m, 'b-', linewidth=2, label='m-測地線 (α=-1)\nη空間で直線')
    
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('t')
    ax2.set_ylabel('θ（自然パラメータ）')
    ax2.set_title('自然パラメータ空間での測地線')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 右図：確率分布の変化
    ax3 = axes[2]
    
    t_samples = [0, 0.25, 0.5, 0.75, 1.0]
    x_vals = [0, 1]
    
    for i, t_val in enumerate(t_samples):
        idx = int(t_val * 99)
        
        # m-geodesic の分布
        p_m_val = p_m[idx]
        ax3.bar([i-0.2], [1-p_m_val], width=0.35, bottom=0, color='blue', alpha=0.5)
        ax3.bar([i-0.2], [p_m_val], width=0.35, bottom=1-p_m_val, color='blue', alpha=0.8)
        
        # e-geodesic の分布
        p_e_val = p_e[idx]
        ax3.bar([i+0.2], [1-p_e_val], width=0.35, bottom=0, color='red', alpha=0.5)
        ax3.bar([i+0.2], [p_e_val], width=0.35, bottom=1-p_e_val, color='red', alpha=0.8)
    
    ax3.set_xticks(range(5))
    ax3.set_xticklabels(['t=0', 't=0.25', 't=0.5', 't=0.75', 't=1'])
    ax3.set_ylabel('確率')
    ax3.set_title('分布の補間\n青: m-測地線, 赤: e-測地線')
    ax3.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("【α-測地線の特徴】")
    print(f"始点: Ber({p1}), θ = {theta1:.3f}")
    print(f"終点: Ber({p2}), θ = {theta2:.3f}")
    print()
    print("m-測地線 (α=-1):")
    print(f"  中点: p = {(p1+p2)/2:.3f} (算術平均)")
    print()
    print("e-測地線 (α=1):")
    print(f"  中点: p = {theta_to_p((theta1+theta2)/2):.3f} (自然パラメータの算術平均)")

visualize_alpha_connections()

### 4.4 曲率の可視化

In [ ]:
def visualize_curvature():
    """正規分布多様体の曲率"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 正規分布多様体のスカラー曲率: R = -1 (一定の負曲率)
    # これは双曲平面と同型
    
    # 左図：曲率の説明
    ax1 = axes[0]
    
    # 双曲平面（ポアンカレ半平面モデル）
    # ds² = (dx² + dy²) / y²
    # これは正規分布のFisher計量 ds² = dμ²/σ² + 2dσ²/σ² と類似
    
    # 測地線（半円）を描画
    for x_center in [-1, 0, 1, 2]:
        for r in [0.5, 1.0, 1.5]:
            theta = np.linspace(0, np.pi, 100)
            x = x_center + r * np.cos(theta)
            y = r * np.sin(theta)
            ax1.plot(x, y, 'b-', alpha=0.5, linewidth=1)
    
    # 垂直線（測地線）
    for x_c in np.linspace(-2, 3, 6):
        ax1.axvline(x_c, color='blue', alpha=0.3, linewidth=1)
    
    ax1.set_xlim(-2, 3)
    ax1.set_ylim(0, 2)
    ax1.set_xlabel('μ')
    ax1.set_ylabel('σ')
    ax1.set_title('正規分布多様体の測地線\n（ポアンカレ半平面モデル）\nスカラー曲率 R = -1')
    ax1.grid(True, alpha=0.3)
    
    # 右図：測地三角形と角度欠損
    ax2 = axes[1]
    
    # 3点
    p1 = np.array([0, 1])
    p2 = np.array([1, 0.5])
    p3 = np.array([-0.5, 0.5])
    
    # 測地線（半円として近似）
    def geodesic_arc(pa, pb, n=50):
        """2点間の測地線（半円）を計算"""
        # 簡略化: 直線で近似（正確には半円）
        t = np.linspace(0, 1, n)
        return pa + t[:, None] * (pb - pa)
    
    arc1 = geodesic_arc(p1, p2)
    arc2 = geodesic_arc(p2, p3)
    arc3 = geodesic_arc(p3, p1)
    
    ax2.plot(arc1[:, 0], arc1[:, 1], 'b-', linewidth=2)
    ax2.plot(arc2[:, 0], arc2[:, 1], 'g-', linewidth=2)
    ax2.plot(arc3[:, 0], arc3[:, 1], 'r-', linewidth=2)
    
    ax2.plot([p1[0], p2[0], p3[0]], [p1[1], p2[1], p3[1]], 'ko', markersize=10)
    
    ax2.set_xlabel('μ')
    ax2.set_ylabel('σ')
    ax2.set_title('負曲率空間での測地三角形\n内角の和 < π')
    ax2.set_xlim(-1, 2)
    ax2.set_ylim(0.2, 1.5)
    ax2.grid(True, alpha=0.3)
    
    # 注釈
    ax2.text(0.5, 0.3, '負曲率: 三角形の\n内角の和 < 180°', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("【正規分布多様体の曲率】")
    print("- スカラー曲率 R = -1（一定の負曲率）")
    print("- 双曲平面（ポアンカレ半平面）と等長")
    print("- 測地線は半円または垂直線")
    print("- 測地三角形の内角の和 < π")

visualize_curvature()

## 5. 具体例

### 例1：Christoffel記号の計算

In [ ]:
def calculate_christoffel_gaussian():
    """正規分布のChristoffel記号を計算"""
    print("【正規分布のChristoffel記号】")
    print()
    print("Fisher計量: g = diag(1/σ², 2/σ²)")
    print("逆計量: g⁻¹ = diag(σ², σ²/2)")
    print()
    print("計量の偏微分:")
    print("  ∂g_{μμ}/∂μ = 0")
    print("  ∂g_{μμ}/∂σ = -2/σ³")
    print("  ∂g_{σσ}/∂μ = 0")
    print("  ∂g_{σσ}/∂σ = -4/σ³")
    print()
    print("Christoffel記号の公式:")
    print("  Γᵏᵢⱼ = (1/2) gᵏˡ (∂gᵢˡ/∂xʲ + ∂gⱼˡ/∂xⁱ - ∂gᵢⱼ/∂xˡ)")
    print()
    print("非ゼロ成分:")
    print("  Γᵘμσ = Γᵘσμ = (1/2)σ²·(-2/σ³) = -1/σ")
    print("  Γˢμμ = -(1/2)(σ²/2)·(-2/σ³) = 1/(2σ) ... (訂正: 1/σ)")
    print("  Γˢσσ = (1/2)(σ²/2)·(-4/σ³) = -1/σ")
    print()
    print("測地線方程式:")
    print("  d²μ/dt² - (2/σ)(dμ/dt)(dσ/dt) = 0")
    print("  d²σ/dt² + (1/σ)(dμ/dt)² - (1/σ)(dσ/dt)² = 0")

calculate_christoffel_gaussian()

### 例2：平行移動の計算

In [ ]:
def demonstrate_parallel_transport():
    """曲線に沿った平行移動"""
    print("【平行移動の方程式】")
    print()
    print("曲線 γ(t) に沿ったベクトル V の平行移動:")
    print("  dV^k/dt + Γᵏᵢⱼ (dγⁱ/dt) Vʲ = 0")
    print()
    
    # 正規分布多様体での平行移動
    def parallel_transport_ode(V, t, gamma_func, dgamma_func, christoffel_func):
        """平行移動の微分方程式"""
        gamma = gamma_func(t)
        dgamma = dgamma_func(t)
        Gamma = christoffel_func(gamma)
        
        dV = np.zeros(2)
        for k in range(2):
            for i in range(2):
                for j in range(2):
                    dV[k] -= Gamma[k, i, j] * dgamma[i] * V[j]
        return dV
    
    # 例: σ = 1 の水平線に沿った平行移動
    def gamma(t):
        return np.array([t, 1])  # μ = t, σ = 1
    
    def dgamma(t):
        return np.array([1, 0])  # dμ/dt = 1, dσ/dt = 0
    
    def christoffel(p):
        sigma = p[1]
        Gamma = np.zeros((2, 2, 2))
        Gamma[0, 0, 1] = -1/sigma
        Gamma[0, 1, 0] = -1/sigma
        Gamma[1, 0, 0] = 1/sigma
        Gamma[1, 1, 1] = -1/sigma
        return Gamma
    
    # 初期ベクトル
    V0 = np.array([0, 1])  # σ方向の単位ベクトル
    
    t_span = np.linspace(0, 2, 100)
    V_solution = odeint(parallel_transport_ode, V0, t_span, 
                        args=(gamma, dgamma, christoffel))
    
    print("例: σ = 1 の水平線に沿った平行移動")
    print(f"初期ベクトル V(0) = {V0}")
    print(f"移動後 V(2) = [{V_solution[-1, 0]:.4f}, {V_solution[-1, 1]:.4f}]")
    print()
    print("この場合、V は変化しない（経路に沿ってΓの寄与が相殺）")
    
    # 可視化
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 経路
    ax.plot(t_span, np.ones_like(t_span), 'b-', linewidth=2, label='経路 γ(t)')
    
    # ベクトルの変化
    skip = 10
    for i in range(0, len(t_span), skip):
        ax.arrow(t_span[i], 1, V_solution[i, 0]*0.2, V_solution[i, 1]*0.2,
                 head_width=0.03, head_length=0.02, fc='red', ec='red')
    
    ax.set_xlabel('μ')
    ax.set_ylabel('σ')
    ax.set_title('σ = 1 に沿った平行移動')
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(0.5, 1.5)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.show()

demonstrate_parallel_transport()

### 例3：双対接続

In [ ]:
def demonstrate_dual_connections():
    """双対接続の関係"""
    print("【双対接続】")
    print()
    print("定義: 接続 ∇ と ∇* が計量 g に関して双対であるとは、")
    print("  X g(Y, Z) = g(∇_X Y, Z) + g(Y, ∇*_X Z)")
    print("が成り立つこと。")
    print()
    print("【α-接続の双対性】")
    print("∇^(α) と ∇^(-α) は Fisher 計量に関して双対")
    print()
    print("特に重要な双対ペア:")
    print("  e-接続 ∇^(1)  ↔  m-接続 ∇^(-1)")
    print()
    print("【情報幾何での意味】")
    print("- e-測地線: 指数型分布族の自然パラメータで直線")
    print("- m-測地線: 混合族の期待値パラメータで直線")
    print("- この双対性が情報幾何の中心的構造")
    print()
    print("【Christoffel記号の関係】")
    print("Γ^(α)ₖᵢⱼ + Γ^(-α)ₖⱼᵢ = ∂gᵢⱼ/∂xᵏ")
    print("（Levi-Civita接続の場合: Γ^(0)ₖᵢⱼ = Γ^(0)ₖⱼᵢ, 自己双対）")

demonstrate_dual_connections()

## 6. 他の概念との関係

### 前の節との繋がり
- **リーマン計量 (§1.5)**: Levi-Civita接続は計量から決まる
- **ベクトル場 (§1.4)**: 共変微分はベクトル場の微分を一般化

### 情報幾何との関連

| 概念 | 一般の微分幾何 | 情報幾何 |
|:---|:---|:---|
| 接続 | $\nabla$ | α-接続 $\nabla^{(\alpha)}$ |
| Christoffel記号 | $\Gamma^k_{ij}$ | $\Gamma^{(\alpha)k}_{ij}$ |
| 測地線 | 最短経路 | α-測地線 |
| 双対接続 | 一般には存在しない | e-接続 ↔ m-接続 |

### 重要な関係式

**α-接続のChristoffel記号**:
$$\Gamma^{(\alpha)k}_{ij} = \Gamma^{(0)k}_{ij} - \frac{\alpha}{2} g^{kl} T_{ijl}$$

**歪度テンソル**:
$$T_{ijk} = E\left[ \partial_i \ell \cdot \partial_j \ell \cdot \partial_k \ell \right]$$

**双対性条件**:
$$g(\nabla^{(\alpha)}_X Y, Z) + g(Y, \nabla^{(-\alpha)}_X Z) = X g(Y, Z)$$

## 7. 演習問題

### Q1. 共変微分の計算

平面上でユークリッド計量を使ったとき、Christoffel記号がすべて0であることを確認せよ。

<details>
<summary>解答を見る</summary>

ユークリッド計量: $g_{ij} = \delta_{ij}$ (定数)

$$\Gamma^k_{ij} = \frac{1}{2} g^{kl} \left( \frac{\partial g_{il}}{\partial x^j} + \frac{\partial g_{jl}}{\partial x^i} - \frac{\partial g_{ij}}{\partial x^l} \right) = 0$$

計量が定数なので、すべての偏微分が0。

</details>

### Q2. ベルヌーイ分布のe-測地線

Ber(0.2) から Ber(0.8) への e-測地線上の t=0.5 での分布を求めよ。

In [ ]:
# Q2検証
def p_to_theta(p):
    return np.log(p / (1 - p))

def theta_to_p(theta):
    return 1 / (1 + np.exp(-theta))

p1, p2 = 0.2, 0.8
theta1, theta2 = p_to_theta(p1), p_to_theta(p2)

# e-測地線の中点（θで線形補間）
theta_mid = 0.5 * theta1 + 0.5 * theta2
p_mid = theta_to_p(theta_mid)

print(f"θ₁ = {theta1:.4f}, θ₂ = {theta2:.4f}")
print(f"θ_mid = {theta_mid:.4f}")
print(f"e-測地線の中点: p = {p_mid:.4f}")
print(f"（参考: 算術平均は p = {(p1+p2)/2:.4f}）")

### Q3. 曲率と平行移動

正規分布多様体のスカラー曲率が $R = -1$ であることから、小さな閉曲線に沿った平行移動でベクトルがどれだけ回転するか説明せよ。

<details>
<summary>解答を見る</summary>

ガウス・ボネの定理より、閉曲線 $C$ に沿った平行移動による回転角 $\Delta\phi$ は：

$$\Delta\phi = \iint_S K \, dA$$

ここで $K$ はガウス曲率（2次元では $K = R/2 = -1/2$）、$S$ は $C$ が囲む領域、$dA$ はFisher計量による面積要素。

負曲率なので、反時計回りの経路では時計回りに回転する。

</details>

## 8. 参考：使用したプロンプト

```
アファイン接続と共変微分の概念を、「曲がった空間での微分」という
観点から直感的に説明してください。ユークリッド空間との違いを強調して。
```

```
正規分布多様体のChristoffel記号を、Fisher計量から計算する過程を
詳細に示してください。
```

```
情報幾何のα-接続について説明してください。特にe-接続とm-接続の
双対性と、それぞれの測地線の特徴を教えてください。
```

```
球面上の平行移動でベクトルが回転することを、Pythonで可視化して
ください。北極から三角形の経路を一周する例を示してください。
```

---
**次のステップ**: 第2章 `ch02_statistical_models/` へ（統計モデルと情報幾何）